In [ ]:
import tensorflow as tf
from tensorflow.keras import layers
import numpy as np
import matplotlib.pyplot as plt

In [ ]:

# Load MNIST dataset
(X_train, _), (_, _) = tf.keras.datasets.mnist.load_data()
X_train = X_train / 127.5 - 1.0  # Normalize to [-1, 1]
X_train = np.expand_dims(X_train, axis=-1)  # Add channel dimension

# Parameters
latent_dim = 100
img_shape = X_train.shape[1:]

# Generator
def build_generator():
    model = tf.keras.Sequential([
        layers.Input(shape=(latent_dim,)),
        layers.Dense(128, activation="relu"),
        layers.BatchNormalization(),
        layers.Dense(256, activation="relu"),
        layers.BatchNormalization(),
        layers.Dense(512, activation="relu"),
        layers.BatchNormalization(),
        layers.Dense(int(np.prod(img_shape)), activation="tanh"),
        layers.Reshape(img_shape)
    ])
    return model

# Discriminator
def build_discriminator():
    model = tf.keras.Sequential([
        layers.Flatten(input_shape=img_shape),
        layers.Dense(512, activation="relu"),
        layers.Dense(256, activation="relu"),
        layers.Dense(1, activation="sigmoid")
    ])
    return model

# Build and compile models
generator = build_generator()
discriminator = build_discriminator()
discriminator.compile(optimizer=tf.keras.optimizers.Adam(0.0002, 0.5), loss="binary_crossentropy", metrics=["accuracy"])

# GAN model
discriminator.trainable = False
gan_input = layers.Input(shape=(latent_dim,))
generated_img = generator(gan_input)
validity = discriminator(generated_img)
gan = tf.keras.Model(gan_input, validity)
gan.compile(optimizer=tf.keras.optimizers.Adam(0.0002, 0.5), loss="binary_crossentropy")

# Training function
def train_gan(epochs, batch_size, save_interval):
    valid = np.ones((batch_size, 1))
    fake = np.zeros((batch_size, 1))

    for epoch in range(epochs):
        # Train Discriminator
        idx = np.random.randint(0, X_train.shape[0], batch_size)
        real_imgs = X_train[idx]

        noise = np.random.normal(0, 1, (batch_size, latent_dim))
        fake_imgs = generator.predict(noise)

        d_loss_real = discriminator.train_on_batch(real_imgs, valid)
        d_loss_fake = discriminator.train_on_batch(fake_imgs, fake)
        d_loss = 0.5 * np.add(d_loss_real, d_loss_fake)

        # Train Generator
        noise = np.random.normal(0, 1, (batch_size, latent_dim))
        g_loss = gan.train_on_batch(noise, valid)

        # Print progress
        print(f"{epoch + 1}/{epochs} [D loss: {d_loss[0]} | D acc: {100 * d_loss[1]:.2f}%] [G loss: {g_loss}]")

        # Save generated images
        if epoch % save_interval == 0:
            save_generated_images(epoch)

# Save generated images
def save_generated_images(epoch, examples=25, dim=(5, 5), figsize=(10, 10)):
    noise = np.random.normal(0, 1, (examples, latent_dim))
    generated_imgs = generator.predict(noise)
    generated_imgs = 0.5 * generated_imgs + 0.5  # Rescale to [0, 1]

    plt.figure(figsize=figsize)
    for i in range(examples):
        plt.subplot(dim[0], dim[1], i + 1)
        plt.imshow(generated_imgs[i, :, :, 0], cmap="gray")
        plt.axis("off")
    plt.tight_layout()
    plt.savefig(f"generated_{epoch}.png")
    plt.close()

# Train GAN
train_gan(epochs=500, batch_size=64, save_interval=1000)

11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


/usr/local/lib/python3.13/dist-packages/keras/src/layers/reshaping/flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step 


/usr/local/lib/python3.13/dist-packages/keras/src/backend/tensorflow/trainer.py:86: UserWarning: The model does not have any trainable weights.
  warnings.warn("The model does not have any trainable weights.")


1/500 [D loss: 1.4343215227127075 | D acc: 24.61%] [G loss: 1.196678638458252]
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 129ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step
2/500 [D loss: 1.2624051570892334 | D acc: 39.45%] [G loss: 1.2012434005737305]
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step
3/500 [D loss: 1.2075128555297852 | D acc: 40.83%] [G loss: 1.168203353881836]
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step
4/500 [D loss: 1.19588041305542 | D acc: 41.07%] [G loss: 1.152768611907959]
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step
5/500 [D loss: 1.1888325214385986 | D acc: 40.77%] [G loss: 1.1506465673446655]
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step
6/500 [D loss: 1.1840341091156006 | D acc: 40.61%] [G loss: 1.139503002166748]
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step
7/500 [D loss: 1.1760368347167969 | D acc: 40.46%] [G loss: 1.1240952014923096]
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step
8/500 [D loss: 1.1725465059280396 | D acc: 40.04%] [G loss: 1.1103260517120361]
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step
9/500 [D loss: 1.174231